# DP Single-Phase Switch State-Space Extraction

This notebook validates state-space extraction for `DP::Ph1::Switch`.

The circuit contains a voltage source and an RL line feeding two parallel
resistive load branches. The first load is always connected. The second load
is connected through a switch.

The notebook checks that:

- the switch contributes no extraction states,
- an open switch matches an ordinary resistor with the switch open resistance,
- a closed switch matches an ordinary resistor with the switch closed resistance,
- the extracted matrix updates after an open-to-closed event,
- both precomputed switch matrices and system-matrix recomputation give the
  closed-configuration matrix,
- the extracted eigenvalues agree with the analytical native-DP eigenvalues.

Only assertions and printed output are used.

In [ ]:
import numpy as np
import dpsimpy

dp = dpsimpy.dp
ph1 = dp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis

## Parameters

In [ ]:
time_step = 100e-6
final_time = 5.0 * time_step
switch_time = 2.0 * time_step

frequency = 50.0
source_voltage = 1.0 * np.exp(1j * 0.0)

line_resistance = 0.5
line_inductance = 0.05

load1_resistance = 20.0
load2_resistance = 10.0

switch_open_resistance = 1e4
switch_closed_resistance = 0.1

matrix_tolerance = 1e-12
eigenvalue_tolerance = 1e-12
minimum_configuration_difference = 1e-6

## Helper functions

In [ ]:
def map_continuous_to_discrete(lam):
    return (2.0 + time_step * lam) / (2.0 - time_step * lam)


def equivalent_load_resistance(switch_resistance):
    switched_branch = switch_resistance + load2_resistance
    return load1_resistance * switched_branch / (load1_resistance + switched_branch)


def expected_discrete_eigenvalues(switch_resistance):
    equivalent_load = equivalent_load_resistance(switch_resistance)
    damping = (line_resistance + equivalent_load) / line_inductance
    omega_shift = 2.0 * np.pi * frequency

    lambda_dp = -damping - 1j * omega_shift
    z = map_continuous_to_discrete(lambda_dp)

    return np.array([z, np.conj(z)], dtype=np.complex128)


def sort_complex(values):
    return np.array(
        sorted(
            np.asarray(values).reshape(-1),
            key=lambda value: (
                round(float(value.imag), 12),
                round(float(value.real), 12),
            ),
        ),
        dtype=np.complex128,
    )


def get_extraction_results(simulation):
    extractor = simulation.get_state_space_extractor()

    modal_analysis = StateSpaceModalAnalysis(extractor)
    modal_analysis.update()

    return {
        "state_count": extractor.get_state_count(),
        "state_names": list(modal_analysis.get_state_names()),
        "Ad": np.array(
            extractor.get_discrete_state_matrix(),
            dtype=float,
            copy=True,
        ),
        "z": np.array(
            modal_analysis.get_discrete_eigenvalues(),
            dtype=np.complex128,
            copy=True,
        ).reshape(-1),
        "lambda": np.array(
            modal_analysis.get_continuous_eigenvalues(),
            dtype=np.complex128,
            copy=True,
        ).reshape(-1),
    }

## Switch model

In [ ]:
def run_switch_model(
    case_name,
    initially_closed,
    add_closing_event=False,
    recompute_system_matrix=False,
):
    source_node = dp.SimNode("n1")
    line_internal_node = dp.SimNode("n2")
    load_node = dp.SimNode("nLoad")
    switched_load_node = dp.SimNode("nSwitchedLoad")

    source = ph1.VoltageSource("VS")
    source.set_parameters(source_voltage, frequency)

    line_resistor = ph1.Resistor("RLine")
    line_resistor.set_parameters(line_resistance)

    line_inductor = ph1.Inductor("LLine")
    line_inductor.set_parameters(line_inductance)

    load1 = ph1.Resistor("RLoad1")
    load1.set_parameters(load1_resistance)

    load_switch = ph1.Switch("LoadSwitch")
    load_switch.set_parameters(
        switch_open_resistance,
        switch_closed_resistance,
        initially_closed,
    )

    load2 = ph1.Resistor("RLoad2")
    load2.set_parameters(load2_resistance)

    source.connect([dp.SimNode.gnd, source_node])
    line_resistor.connect([source_node, line_internal_node])
    line_inductor.connect([line_internal_node, load_node])
    load1.connect([load_node, dp.SimNode.gnd])
    load_switch.connect([load_node, switched_load_node])
    load2.connect([switched_load_node, dp.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [source_node, line_internal_node, load_node, switched_load_node],
        [source, line_resistor, line_inductor, load1, load_switch, load2],
    )

    simulation = Simulation(f"DP_Ph1_Switch_StateSpaceExtraction_{case_name}")
    simulation.set_system(system)
    simulation.set_domain(Domain.DP)
    simulation.set_solver(Solver.MNA)
    simulation.set_time_step(time_step)
    simulation.set_final_time(final_time)
    simulation.do_state_space_extraction(True)

    if recompute_system_matrix:
        simulation.do_system_matrix_recomputation(True)

    if add_closing_event:
        simulation.add_event(
            dpsimpy.event.SwitchEvent(
                switch_time,
                load_switch,
                True,
            )
        )

    simulation.run()
    return get_extraction_results(simulation)

## Equivalent-resistor reference model

In [ ]:
def run_resistor_reference(case_name, switch_resistance):
    source_node = dp.SimNode("n1")
    line_internal_node = dp.SimNode("n2")
    load_node = dp.SimNode("nLoad")
    switched_load_node = dp.SimNode("nSwitchedLoad")

    source = ph1.VoltageSource("VS")
    source.set_parameters(source_voltage, frequency)

    line_resistor = ph1.Resistor("RLine")
    line_resistor.set_parameters(line_resistance)

    line_inductor = ph1.Inductor("LLine")
    line_inductor.set_parameters(line_inductance)

    load1 = ph1.Resistor("RLoad1")
    load1.set_parameters(load1_resistance)

    switch_reference = ph1.Resistor("LoadSwitch")
    switch_reference.set_parameters(switch_resistance)

    load2 = ph1.Resistor("RLoad2")
    load2.set_parameters(load2_resistance)

    source.connect([dp.SimNode.gnd, source_node])
    line_resistor.connect([source_node, line_internal_node])
    line_inductor.connect([line_internal_node, load_node])
    load1.connect([load_node, dp.SimNode.gnd])
    switch_reference.connect([load_node, switched_load_node])
    load2.connect([switched_load_node, dp.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [source_node, line_internal_node, load_node, switched_load_node],
        [
            source,
            line_resistor,
            line_inductor,
            load1,
            switch_reference,
            load2,
        ],
    )

    simulation = Simulation(
        f"DP_Ph1_Switch_StateSpaceExtraction_{case_name}_ResistorReference"
    )
    simulation.set_system(system)
    simulation.set_domain(Domain.DP)
    simulation.set_solver(Solver.MNA)
    simulation.set_time_step(time_step)
    simulation.set_final_time(final_time)
    simulation.do_state_space_extraction(True)
    simulation.run()

    return get_extraction_results(simulation)

## Run all cases

In [ ]:
open_switch = run_switch_model("Open", initially_closed=False)
open_reference = run_resistor_reference(
    "Open",
    switch_open_resistance,
)

closed_switch = run_switch_model("Closed", initially_closed=True)
closed_reference = run_resistor_reference(
    "Closed",
    switch_closed_resistance,
)

switched_precomputed = run_switch_model(
    "Event_Precomputed",
    initially_closed=False,
    add_closing_event=True,
    recompute_system_matrix=False,
)

switched_recomputed = run_switch_model(
    "Event_Recomputed",
    initially_closed=False,
    add_closing_event=True,
    recompute_system_matrix=True,
)

## Assertions and output

In [ ]:
expected_state_names = ["LLine_re", "LLine_im"]
expected_state_count = len(expected_state_names)


def validate_equivalent(label, actual, reference):
    assert (
        actual["state_count"] == reference["state_count"]
    ), f"{label}: state counts differ."
    assert (
        actual["state_names"] == reference["state_names"]
    ), f"{label}: state names differ."

    np.testing.assert_allclose(
        actual["Ad"],
        reference["Ad"],
        rtol=0.0,
        atol=matrix_tolerance,
        err_msg=f"{label}: state matrices differ",
    )

    np.testing.assert_allclose(
        sort_complex(actual["z"]),
        sort_complex(reference["z"]),
        rtol=0.0,
        atol=eigenvalue_tolerance,
        err_msg=f"{label}: eigenvalues differ",
    )

    matrix_error = np.max(np.abs(actual["Ad"] - reference["Ad"]))
    eigenvalue_error = np.max(
        np.abs(sort_complex(actual["z"]) - sort_complex(reference["z"]))
    )

    print(label)
    print(f"  Maximum absolute Ad difference: {matrix_error:.12e}")
    print(f"  Maximum eigenvalue difference:  {eigenvalue_error:.12e}")


assert open_switch["state_count"] == expected_state_count
assert open_switch["state_names"] == expected_state_names

validate_equivalent(
    "Open switch vs. open-resistance reference",
    open_switch,
    open_reference,
)
validate_equivalent(
    "Closed switch vs. closed-resistance reference",
    closed_switch,
    closed_reference,
)
validate_equivalent(
    "Open-to-closed event with precomputed switch matrices",
    switched_precomputed,
    closed_switch,
)
validate_equivalent(
    "Open-to-closed event with system-matrix recomputation",
    switched_recomputed,
    closed_switch,
)

configuration_difference = np.max(np.abs(open_switch["Ad"] - closed_switch["Ad"]))
assert configuration_difference >= minimum_configuration_difference, (
    "Open and closed switch configurations are not sufficiently " "distinguishable."
)

np.testing.assert_allclose(
    sort_complex(open_switch["z"]),
    sort_complex(expected_discrete_eigenvalues(switch_open_resistance)),
    rtol=0.0,
    atol=eigenvalue_tolerance,
    err_msg="Open-switch eigenvalues do not match the analytical reference.",
)

np.testing.assert_allclose(
    sort_complex(closed_switch["z"]),
    sort_complex(expected_discrete_eigenvalues(switch_closed_resistance)),
    rtol=0.0,
    atol=eigenvalue_tolerance,
    err_msg="Closed-switch eigenvalues do not match the analytical reference.",
)

print("\n" + "=" * 60)
print("DP Ph1 switch state-space extraction validation")
print("=" * 60)
print(
    f"Open/closed maximum absolute Ad difference: " f"{configuration_difference:.12e}"
)

print("\nExtraction states:")
for index, state_name in enumerate(open_switch["state_names"]):
    print(f"  [{index}] {state_name}")

print("\nOpen-switch discrete-time eigenvalues:")
for index, eigenvalue in enumerate(open_switch["z"]):
    print(f"  [{index}] {eigenvalue}")

print("\nClosed-switch discrete-time eigenvalues:")
for index, eigenvalue in enumerate(closed_switch["z"]):
    print(f"  [{index}] {eigenvalue}")

print("\nAll DP Ph1 switch state-space extraction checks passed.")